In [1]:
import requests
import json
import os
from dotenv import load_dotenv
import pandas as pd
load_dotenv()

True

## setup Poliscope API

In [2]:
poliscope_api_key = os.getenv("POLISCOPE_API_KEY")
poliscope_api_url = os.getenv("POLISCOPE_API_URL")
# API headers
poliscope_headers = {
    "Authorization": f"Bearer {poliscope_api_key}",
    "x-client": "helen-integration"
}

## get all entities

Entities sind die angebundenen RIS. Mit diesem Code-Block ziehen wir uns alle entities mit Metadaten und ids.

Diese Liste ist nötig, um später gezielt für bestimmte Kommunen Daten zu finden, oder Rechercheergebnisse passend zu filtern.


In [3]:
# Alle Items sammeln
all_entities = []

# Paginierungsparameter
#achtung, tatsächlich ist gerade auf 300 Limit
limit = 500
offset = 0
total = 16079

# Durch alle Pages iterieren
while offset < total:
    response = requests.get(
        url=f"{poliscope_api_url}/entities",
        headers=poliscope_headers,
        params={
            "limit": limit,
            "offset": offset,
            "detail": "standard"
        }
    )
    
    if response.status_code == 200:
        data = response.json()
        items = data.get("data", [])
        all_entities.extend(items)  # Alle Items zur Liste hinzufügen
        
        print(f"Downloaded {offset + len(items)} / {total} items")
        offset += limit
    else:
        print(f"Error: {response.status_code}")
        break

Downloaded 500 / 16079 items
Downloaded 1000 / 16079 items
Downloaded 1500 / 16079 items
Downloaded 2000 / 16079 items
Downloaded 2500 / 16079 items
Downloaded 3000 / 16079 items
Downloaded 3500 / 16079 items
Downloaded 4000 / 16079 items
Downloaded 4500 / 16079 items
Downloaded 5000 / 16079 items
Downloaded 5500 / 16079 items
Downloaded 6000 / 16079 items
Downloaded 6500 / 16079 items
Downloaded 7000 / 16079 items
Downloaded 7500 / 16079 items
Downloaded 8000 / 16079 items
Downloaded 8500 / 16079 items
Downloaded 9000 / 16079 items
Downloaded 9500 / 16079 items
Downloaded 10000 / 16079 items
Downloaded 10500 / 16079 items


KeyboardInterrupt: 

In [27]:
entities_df = pd.DataFrame(all_entities)
entities_df

,id,name,level,location,parents,population,area,postalCode,city,street,ris
0,072355007001,Aach,60,"{'lat': 49.789424896240234, 'lon': 6.590655803...","[{'id': '072355007', 'name': 'Trier-Land', 'le...",1142.0,6.96000,54295,Trier,Gartenfeldstraße 12,None
1,083355001001,"Aach, Stadt",60,"{'lat': 47.84281921386719, 'lon': 8.8508443832...","[{'id': '083355001', 'name': 'VVG der Stadt En...",2304.0,10.68000,78234,Engen,Hauptstraße 11,"{'id': 3206, 'status': 'active', 'urls': ['htt..."
2,053340002002,"Aachen, Stadt",60,"{'lat': 50.775428771972656, 'lon': 6.081491947...","[{'id': '053340002', 'name': 'Aachen, Stadt', ...",249070.0,160.85001,52058,Aachen,Markt,None
3,053340002,"Aachen, Stadt",50,"{'lat': 50.7764646874706, 'lon': 6.08396351207...","[{'id': '05334', 'name': 'Städteregion Aachen'...",249070.0,NaN,52058,Aachen,Markt,"{'id': 5373, 'status': 'active', 'urls': ['htt..."
4,081365001088,"Aalen, Stadt",60,"{'lat': 48.83597183227539, 'lon': 10.089909553...","[{'id': '081365001', 'name': 'VVG der Stadt Aa...",68351.0,146.58000,73430,Aalen,Marktplatz 30,"{'id': 1228, 'status': 'active', 'urls': ['htt..."
...,...,...,...,...,...,...,...,...,...,...,...
16074,082255006113,Zwingenberg,60,"{'lat': 49.41682434082031, 'lon': 9.0412893295...","[{'id': '082255006', 'name': 'GVV Neckargerach...",680.0,4.72000,69437,Neckargerach,Hauptstraße 25,None
16075,064310022022,"Zwingenberg, Stadt",60,"{'lat': 49.72316360473633, 'lon': 8.6126108169...","[{'id': '064310022', 'name': 'Zwingenberg, Sta...",7202.0,5.66000,64673,Zwingenberg,Untergasse 16,None
16076,064310022,"Zwingenberg, Stadt",50,"{'lat': 49.723706, 'lon': 8.61302}","[{'id': '06431', 'name': 'Bergstraße', 'level'...",7202.0,NaN,64673,Zwingenberg,Untergasse 16,"{'id': 4627, 'status': 'active', 'urls': ['htt..."
16077,145215140,Zwönitz,50,"{'lat': 50.630205, 'lon': 12.813659}","[{'id': '14521', 'name': 'Erzgebirgskreis', 'l...",14601.0,109.98000,08297,Zwönitz,Markt 6,None


In [ ]:
entities_df.to_csv("./data/metadata/all_entities.csv", index=False)

# Großstädte filtern

Basierend auf dem entities-dataframe können wir bestimmte Gruppen von Städten filtern, etwa Großstädte. Dazu können auch andere Datenquellen herangezogen werden, je nachdem welcher Filter gewünscht ist.

10 - Bundesländer

40 - Landkreise

50 - Gemeindeverbände

60 - Gemeinden

PR - Planungsregionen

id entspricht dem ARS Schlüssel

In [55]:
big_cities = entities_df[(entities_df["population"] >= 100000)].copy()
big_cities = big_cities[big_cities["ris"].notnull()]

In [56]:
big_cities = big_cities[big_cities["level"].isin(["60", "50", "10"])]
big_cities = big_cities[~big_cities["parents"].apply(lambda x: x[0]["name"] if x else None).isin(["Berlin, Stadt", "Hamburg, Freie und Hansestadt"])]

In [58]:
big_cities.to_csv("./data/raw/big_cities.csv", index=False)